In [1]:
import numpy as np
import math
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

# =========================================================
# WEEK 8 FUNCTION 6 — UPDATED TO PRODUCE NEXT DATAPOINT (6 d.p.)
# - Data hygiene: clamp to [0,1], merge exact duplicates
# - Surrogate: Deep Ensemble (bootstrap + different seeds)
# - Candidates: Sobol global + Trust-region local
# - Acquisition: Hybrid EI/PI with adaptive xi
# - Output: x_next printed to 6 decimals (and top-K shortlist)
# =========================================================

# -----------------------------
# 1) Data (as provided)
# -----------------------------
X_raw = np.array([
 [0.7281861, 0.15469257, 0.73255167, 0.69399651, 0.05640131],
 [0.24238435, 0.84409997, 0.5778091, 0.67902128, 0.50195289],
 [0.72952261, 0.7481062, 0.67977464, 0.35655228, 0.67105368],
 [0.77062024, 0.11440374, 0.04677993, 0.64832428, 0.27354905],
 [0.6188123, 0.33180214, 0.18728787, 0.75623847, 0.3288348],
 [0.78495809, 0.91068235, 0.7081201, 0.95922543, 0.0049115],
 [0.14511079, 0.8966846, 0.89632223, 0.72627154, 0.23627199],
 [0.94506907, 0.28845905, 0.97880576, 0.96165559, 0.59801594],
 [0.12572016, 0.86272469, 0.02854433, 0.24660527, 0.75120624],
 [0.75759436, 0.35583141, 0.0165229, 0.4342072, 0.11243304],
 [0.5367969, 0.30878091, 0.41187929, 0.38822518, 0.5225283],
 [0.95773967, 0.23566857, 0.09914585, 0.15680593, 0.07131737],
 [0.6293079, 0.80348368, 0.81140844, 0.04561319, 0.11062446],
 [0.02173531, 0.42808424, 0.83593944, 0.48948866, 0.51108173],
 [0.43934426, 0.69892383, 0.42682022, 0.10947609, 0.87788847],
 [0.25890557, 0.79367771, 0.6421139, 0.19667346, 0.59310318],
 [0.43216593, 0.71561781, 0.3418191, 0.70499988, 0.61496184],
 [0.78287982, 0.53633586, 0.44328356, 0.85969983, 0.01032599],
 [0.9217762, 0.93187122, 0.41487637, 0.59505727, 0.73562569],
 [0.12667892, 0.2914703, 0.06452848, 0.6805146, 0.89281919],
 [1.057739, 1.031871, 1.078805, 1.061655, 0.992819],
 [0.183405, 0.304243, 0.524756, 0.431945, 0.29123 ],
 [0.268807, 0.268756, 0.495982, 0.986904, 0.010463],
 [0.071886, 0.119564, 0.11427 , 0.97486 , 0.062381],
 [0.985053, 0.912856, 0.967723, 0.993021, 0.944062],
 [0.985053, 0.912856, 0.967723, 0.993021, 0.944062],
 [0.289326, 0.014308, 0.819185, 0.769045, 0.018673]
], dtype=float)

y_raw = np.array([
 -0.71426495, -1.20995524, -1.67219994, -1.53605771, -0.82923655,
 -1.24704893, -1.23378638, -1.69434344, -2.57116963, -1.30911635,
 -1.14478485, -1.91267714, -1.62283895, -1.35668211, -2.0184254,
 -1.70255784, -1.29424696, -0.93575656, -2.15576776, -1.74688209,
 -2.868905011263093, -0.9489046340640067, -0.6674108573004914,
 -1.3769650251311083, -2.5104529076756172, -2.5498833751068073, -0.7529123509459484
], dtype=float)

assert X_raw.shape[0] == y_raw.shape[0], "X and y must have the same number of rows."

# -----------------------------
# 2) Device & Seeds
# -----------------------------
RANDOM_SEED = 123
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# 3) Data hygiene (clamp + dedup)
# -----------------------------
def clamp01(X):
    return np.clip(X, 0.0, 1.0)

def dedup_average(X, y, tol=0.0):
    """
    Merge duplicates by averaging y for identical X.
    tol=0.0 -> exact duplicates only.
    """
    if tol > 0:
        X_key = np.round(X / tol) * tol
    else:
        X_key = X.copy()

    keys = [row.tobytes() for row in X_key]
    buckets = {}
    for i, k in enumerate(keys):
        buckets.setdefault(k, []).append(i)

    X_new, y_new = [], []
    for k, idxs in buckets.items():
        X_new.append(X[idxs[0]])
        y_new.append(float(np.mean(y[idxs])))
    return np.array(X_new, dtype=float), np.array(y_new, dtype=float)

X_raw = clamp01(X_raw)
X_raw, y_raw = dedup_average(X_raw, y_raw, tol=0.0)

# -----------------------------
# 4) Scaling
# -----------------------------
x_scaler = StandardScaler()
y_scaler = StandardScaler()
X_scaled = x_scaler.fit_transform(X_raw)
y_scaled = y_scaler.fit_transform(y_raw.reshape(-1, 1)).ravel()

X_tensor_all = torch.tensor(X_scaled, dtype=torch.float32, device=device)
y_tensor_all = torch.tensor(y_scaled.reshape(-1, 1), dtype=torch.float32, device=device)

# -----------------------------
# 5) Surrogate Model
# -----------------------------
class SurrogateNN(nn.Module):
    def __init__(self, input_dim=5, hidden_layers=(128, 64), dropout_p=0.05, use_layernorm=True):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_layers:
            layers.append(nn.Linear(prev, h))
            if use_layernorm:
                layers.append(nn.LayerNorm(h))
            layers.append(nn.GELU())
            layers.append(nn.Dropout(dropout_p))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# -----------------------------
# 6) Training helper
# -----------------------------
def train_with_early_stopping(model, optimizer, criterion, X_train, y_train, X_val, y_val,
                              max_epochs=2000, patience=250, min_epochs=200):
    best_state = None
    best_val = float("inf")
    no_improve = 0

    for ep in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad()
        pred = model(X_train)
        loss = criterion(pred, y_train)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val)
            val_loss = criterion(val_pred, y_val).item()

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            if ep >= min_epochs:
                no_improve += 1

        if ep >= min_epochs and no_improve >= patience:
            break

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    return best_val

# -----------------------------
# 7) Hyperparameter tuning (random search + CV)
# -----------------------------
def sample_log_uniform(rng, lo, hi):
    return float(np.exp(rng.uniform(np.log(lo), np.log(hi))))

def sample_config(rng):
    hidden_choices = [(64, 64), (128, 64), (128, 128), (256, 128), (256, 256)]
    dropout_choices = [0.0, 0.03, 0.05, 0.08, 0.12, 0.18]
    ln_choices = [True, False]

    return {
        "hidden_layers": hidden_choices[rng.integers(0, len(hidden_choices))],
        "dropout": float(dropout_choices[rng.integers(0, len(dropout_choices))]),
        "lr": sample_log_uniform(rng, 3e-4, 4e-3),
        "weight_decay": sample_log_uniform(rng, 1e-7, 3e-4),
        "max_epochs": int(rng.choice([1000, 1400, 1800, 2200])),
        "patience": int(rng.choice([180, 220, 260, 320])),
        "min_epochs": int(rng.choice([150, 200, 250])),
        "xi_base": float(rng.choice([0.0, 0.005, 0.01, 0.02])),
        "use_layernorm": bool(rng.choice(ln_choices)),
        "n_ensemble": int(rng.choice([5, 7]))
    }

def cv_score_config(cfg, X_all, y_all, seed=123, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    criterion = nn.MSELoss()
    vals = []

    for tr_idx, va_idx in kf.split(X_all):
        X_tr = X_all[tr_idx]
        y_tr = y_all[tr_idx]
        X_va = X_all[va_idx]
        y_va = y_all[va_idx]

        model = SurrogateNN(
            input_dim=5,
            hidden_layers=cfg["hidden_layers"],
            dropout_p=cfg["dropout"],
            use_layernorm=cfg["use_layernorm"]
        ).to(device)

        optimizer = optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

        val = train_with_early_stopping(
            model, optimizer, criterion,
            X_tr, y_tr, X_va, y_va,
            max_epochs=cfg["max_epochs"],
            patience=cfg["patience"],
            min_epochs=cfg["min_epochs"]
        )
        vals.append(val)

    return float(np.mean(vals))

def tune_hyperparameters(X_all, y_all, n_trials=30, seed=123):
    rng = np.random.default_rng(seed)
    best_cfg = None
    best_score = float("inf")

    for t in range(1, n_trials + 1):
        cfg = sample_config(rng)
        score = cv_score_config(cfg, X_all, y_all, seed=seed, n_splits=5)

        if score < best_score:
            best_score = score
            best_cfg = cfg

        print(f"Trial {t:02d}/{n_trials} | CV-MSE={score:.6f} | cfg={cfg}")

    return best_cfg, best_score

# -----------------------------
# 8) Deep Ensemble fit + predict
# -----------------------------
def fit_ensemble(cfg, X_all, y_all, base_seed=123):
    models = []
    criterion = nn.MSELoss()
    n = X_all.shape[0]

    for m in range(cfg["n_ensemble"]):
        # bootstrap sample for diversity
        rng = np.random.default_rng(base_seed + 10_000 + m)
        boot = rng.integers(0, n, size=n)
        X_b = X_all[boot]
        y_b = y_all[boot]

        # val split for early stopping
        idx = torch.randperm(n, device=device)
        n_train = max(int(0.85 * n), 1)
        tr_idx, va_idx = idx[:n_train], idx[n_train:]

        model = SurrogateNN(
            input_dim=5,
            hidden_layers=cfg["hidden_layers"],
            dropout_p=cfg["dropout"],
            use_layernorm=cfg["use_layernorm"]
        ).to(device)

        optimizer = optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

        train_with_early_stopping(
            model, optimizer, criterion,
            X_b[tr_idx], y_b[tr_idx],
            X_b[va_idx], y_b[va_idx],
            max_epochs=cfg["max_epochs"],
            patience=cfg["patience"],
            min_epochs=cfg["min_epochs"]
        )

        models.append(model)

    return models

@torch.no_grad()
def predict_ensemble(models, X_candidates):
    X_scaled_cand = x_scaler.transform(X_candidates)
    X_t = torch.tensor(X_scaled_cand, dtype=torch.float32, device=device)

    preds_scaled = []
    for model in models:
        model.eval()
        preds_scaled.append(model(X_t).detach().cpu().numpy())  # [n,1]

    preds_scaled = np.stack(preds_scaled, axis=0).squeeze(-1)  # [E, n]

    # invert scaling per member
    flat = preds_scaled.reshape(-1, 1)
    orig = y_scaler.inverse_transform(flat).reshape(preds_scaled.shape)

    mu = orig.mean(axis=0)
    sigma = orig.std(axis=0)
    return mu, sigma

# -----------------------------
# 9) Acquisition (EI/PI)
# -----------------------------
def normal_pdf(z):
    return np.exp(-0.5 * z**2) / math.sqrt(2 * math.pi)

def normal_cdf(z):
    z = np.asarray(z)
    return 0.5 * (1 + np.vectorize(math.erf)(z / math.sqrt(2)))

def acquisition_ei_pi(mu, sigma, y_best, xi=0.0):
    eps = 1e-9
    sigma = np.maximum(sigma, eps)

    improvement = mu - y_best - xi
    Z = improvement / sigma

    cdf_vals = normal_cdf(Z)
    pdf_vals = normal_pdf(Z)

    ei = improvement * cdf_vals + sigma * pdf_vals
    pi = cdf_vals
    return np.maximum(ei, 0.0), pi

# -----------------------------
# 10) Candidate generation (Sobol + trust-region)
# -----------------------------
def sobol_candidates(n, dim, seed=123):
    engine = torch.quasirandom.SobolEngine(dimension=dim, scramble=True, seed=seed)
    return engine.draw(n).cpu().numpy()

def trust_region_candidates(x_center, n, sigma=0.08, seed=123):
    rng = np.random.default_rng(seed)
    X = x_center.reshape(1, -1) + rng.normal(0.0, sigma, size=(n, x_center.size))
    return np.clip(X, 0.0, 1.0)

def min_dist_filter(X_cand, X_obs, min_dist=1e-3):
    # keep candidates at least min_dist away from every observed point
    diff = X_cand[:, None, :] - X_obs[None, :, :]
    d2 = np.sum(diff * diff, axis=2)
    return np.all(d2 >= (min_dist ** 2), axis=1)

# -----------------------------
# 11) Propose next point (and top-K list), print to 6 decimals
# -----------------------------
def propose_next_points(models, X_obs, y_obs,
                        xi_base=0.01,
                        n_sobol=24000, n_local=8000,
                        random_seed=123,
                        min_dist=1e-3,
                        ei_top_frac=0.05,
                        local_sigma=0.08,
                        top_k=10):

    # Center for local exploitation
    x_best_obs = X_obs[int(np.argmax(y_obs))]

    # Candidates
    X_global = sobol_candidates(n_sobol, dim=5, seed=random_seed)
    X_local = trust_region_candidates(x_best_obs, n_local, sigma=local_sigma, seed=random_seed + 7)
    X_cand = np.vstack([X_global, X_local])

    # Distance filter
    keep = min_dist_filter(X_cand, X_obs, min_dist=min_dist)
    X_cand = X_cand[keep]

    # Predict
    mu, sigma = predict_ensemble(models, X_cand)

    # Adaptive exploration
    y_best = float(np.max(y_obs))
    y_min = float(np.min(y_obs))
    y_range = max(y_best - y_min, 1e-6)

    sigma_med = float(np.median(sigma))
    conf_scale = 1.0 / (1.0 + sigma_med)
    effective_xi = xi_base * y_range * conf_scale

    ei, pi = acquisition_ei_pi(mu, sigma, y_best, xi=effective_xi)

    # Hybrid EI/PI selection
    max_ei = float(np.max(ei))
    thresh = max_ei * (1.0 - ei_top_frac)
    idx_pool = np.where(ei >= thresh)[0]
    if len(idx_pool) == 0:
        best_idx = int(np.argmax(ei))
    else:
        best_idx = int(idx_pool[np.argmax(pi[idx_pool])])

    # Top-K shortlist by EI (non-duplicate already enforced by min_dist)
    top_idx = np.argsort(-ei)[:top_k]

    return {
        "x_next": X_cand[best_idx],
        "mu_next": float(mu[best_idx]),
        "sigma_next": float(sigma[best_idx]),
        "ei_next": float(ei[best_idx]),
        "pi_next": float(pi[best_idx]),
        "x_best_obs": x_best_obs,
        "y_best_obs": y_best,
        "effective_xi": float(effective_xi),
        "max_ei": max_ei,
        "ei_threshold": float(thresh),
        "pool_size": int(len(idx_pool)),
        "n_candidates": int(X_cand.shape[0]),
        "topk": [
            {
                "rank": int(r + 1),
                "x": X_cand[i],
                "mu": float(mu[i]),
                "sigma": float(sigma[i]),
                "ei": float(ei[i]),
                "pi": float(pi[i]),
            }
            for r, i in enumerate(top_idx)
        ]
    }

# -----------------------------
# 12) Local sensitivity (use first ensemble member)
# -----------------------------
def local_sensitivity(model, x_point):
    model.eval()
    x_scaled = x_scaler.transform(x_point.reshape(1, -1))
    x_t = torch.tensor(x_scaled, dtype=torch.float32, device=device, requires_grad=True)

    y_pred = model(x_t)
    y_pred.backward()

    grads = x_t.grad.detach().cpu().numpy().flatten()
    g = np.abs(grads)
    if g.sum() == 0:
        return np.ones_like(g) / len(g)
    return g / g.sum()

# -----------------------------
# 13) Main
# -----------------------------
def main():
    print("\n================ WEEK 8 FUNCTION 6 — NEXT DATA POINT (6 DECIMALS) ================\n")
    print(f"Data after clamp+dedup: n={X_raw.shape[0]} points")

    best_cfg, best_cv = tune_hyperparameters(
        X_tensor_all, y_tensor_all,
        n_trials=30,
        seed=RANDOM_SEED
    )

    print("\n================ BEST TUNED CONFIG ================\n")
    print("Best CV-MSE (scaled y):", best_cv)
    print("Best config:", best_cfg)

    print("\n================ FITTING DEEP ENSEMBLE ================\n")
    ensemble_models = fit_ensemble(best_cfg, X_tensor_all, y_tensor_all, base_seed=RANDOM_SEED)

    details = propose_next_points(
        ensemble_models,
        X_raw,
        y_raw,
        xi_base=best_cfg["xi_base"],
        n_sobol=24000,
        n_local=8000,
        random_seed=RANDOM_SEED,
        min_dist=1e-3,
        ei_top_frac=0.05,
        local_sigma=0.08,
        top_k=10
    )

    x_next_6 = np.round(details["x_next"].astype(float), 6)

    print("\n================ CURRENT BEST OBSERVED ================\n")
    print("x_best =", np.round(details["x_best_obs"].astype(float), 6))
    print("y_best =", round(details["y_best_obs"], 6))

    print("\n================ SETTINGS (Week 8) ================\n")
    print(f"- n_candidates_after_filter = {details['n_candidates']}")
    print(f"- effective_xi = {details['effective_xi']:.6f}")
    print(f"- max_EI = {details['max_ei']:.6f}")
    print(f"- EI threshold (top 5%) = {details['ei_threshold']:.6f}")
    print(f"- pool_size_above_threshold = {details['pool_size']}")

    print("\n================ NEXT DATA POINT (6 DECIMALS) ================\n")
    # This is the value you submit as the next evaluation point
    print("x_next =", x_next_6)
    print("mu(x_next) =", round(details["mu_next"], 6))
    print("sigma(x_next) =", round(details["sigma_next"], 6))
    print("EI =", round(details["ei_next"], 6))
    print("PI =", round(details["pi_next"], 6))

    print("\n================ TOP-10 SHORTLIST (by EI) — 6 DECIMALS ================\n")
    for row in details["topk"]:
        print(
            f"{row['rank']:02d}) x={np.round(row['x'].astype(float), 6)} | "
            f"mu={row['mu']:.6f} | sigma={row['sigma']:.6f} | EI={row['ei']:.6f} | PI={row['pi']:.6f}"
        )

    print("\n================ LOCAL SENSITIVITY (ensemble member #1) ================\n")
    sens = local_sensitivity(ensemble_models[0], details["x_next"])
    for i, s in enumerate(sens, 1):
        print(f"Dim {i}: {s:.3f}")

if __name__ == "__main__":
    main()



================ WEEK 8 FUNCTION 6 — NEXT DATA POINT (6 DECIMALS) ================

Data after clamp+dedup: n=26 points
Trial 01/30 | CV-MSE=0.283848 | cfg={'hidden_layers': (64, 64), 'dropout': 0.12, 'lr': 0.00034487888329125314, 'weight_decay': 5.837380423441397e-07, 'max_epochs': 1400, 'patience': 180, 'min_epochs': 200, 'xi_base': 0.0, 'use_layernorm': True, 'n_ensemble': 7}
Trial 02/30 | CV-MSE=0.272138 | cfg={'hidden_layers': (128, 128), 'dropout': 0.18, 'lr': 0.0006141161355283651, 'weight_decay': 7.085753007178909e-05, 'max_epochs': 2200, 'patience': 320, 'min_epochs': 150, 'xi_base': 0.01, 'use_layernorm': True, 'n_ensemble': 5}
Trial 03/30 | CV-MSE=0.169210 | cfg={'hidden_layers': (128, 64), 'dropout': 0.12, 'lr': 0.0005219050422180717, 'weight_decay': 3.7859148943948664e-05, 'max_epochs': 1000, 'patience': 260, 'min_epochs': 200, 'xi_base': 0.02, 'use_layernorm': False, 'n_ensemble': 5}
Trial 04/30 | CV-MSE=0.254679 | cfg={'hidden_layers': (256, 256), 'dropout': 0.12, 'lr':